# Parallelism in LLM

```{note}
参考：https://zhuanlan.zhihu.com/p/1904506837543420662
```

## 数据并行 (Data Parallel)

* 每张卡都放完整的模型，仅对数据做切分，计算完梯度后需做一次 all-reduce，然后使用优化器更新模型。
* 数据并行在所有并行中效率最高，除了一次 all-reduce 外，没有任何额外通信开销。数据并行的缺点是只能放小模型。

```{figure} ../images/parallel1.webp
```

## 张量并行（Tensor Parallel）

* 张量并行是指对模型内部的参数矩阵做切分，然后利用分块矩阵乘法进行计算得到正确结果。
* 张量并行的优点是能分摊模型到多张卡上，缺点是带来了不小的通信开销，影响训练效率。

```{figure} ../images/parallel2.jpg
```

## 序列并行（Sequence Parallel）

* TP 主要处理的是 attention 和 FFN 维度的并行，主要是对参数矩阵进行切分，而输入还是有冗余（每个 TP 在 attention 和 MLP 有相同的输入）。SP 则是对输入矩阵按 sequence 维度进行切分，体现在 layernom 和 dropout 层。
* SP 与 TP 结合，在进入 attention 和 MLP 时需要做 all-gather，反向时需要做 reduce-scatter；从 attention 和 MLP 出来时需做 reduce-scatter，反向做 all-gather。

```{figure} ../images/parallel3.jpg
```

## Distributed Data Parallel

* DDP 原理：在分类上，DDP 属于 Data Parallel。简单来讲，就是通过提高 batch_size 来增加并行度。
* 为什么快：DDP 通过 Ring-Reduce 的数据交换方法提高了通讯效率，并通过启动多个进程的方式减轻 Python GIL 的限制，从而提高训练速度。

## DeepSpeed ZeRO

传统数据并行中，每个设备会存储完整的模型参数、梯度和优化器状态（存在大量冗余）。ZeRO 的核心是消除这种冗余，将这些数据在多个设备间进行划分，每个设备只保留一部分，从而大幅减少单设备内存压力。

1. **ZeRO-1**（Optimizer State Partitioning）
    * 仅对优化器状态（如 Adam 中的动量和方差）进行划分
    * 每个设备只存储优化器状态的 1/N（N 为数据并行数）
    * 前向 / 反向传播时，每个设备仍保留完整参数和梯度
2. **ZeRO-2**（Gradient Partitioning）
    * 在 ZeRO-1 基础上，增加对梯度的划分
    * 每个设备只存储梯度的 1/N
    * 仍保留完整的模型参数
3. **ZeRO-3**（Parameter Partitioning）
    * 进一步对模型参数进行划分，每个设备只存储 1/N 的参数
    * 前向传播时，通过通信获取当前计算所需的参数（计算后释放）
    * 反向传播时，同样按需获取参数并计算梯度

## FSDP

FSDP 全称 FullyShardedDataParallel， 是Meta 提出的一个针对LLM 训练的解决方案，它是一个数据并行的策略，通过对模型参数（parameters), 梯度（gradients) 和优化器状态（optimizer states) 在多gpu上点切分实现并行。

FSDP受启发于DeepSpeed ZeRO-DP，并进行了进一步的延申和拓展。FSDP包括了NO_SGARD（等效于DDP)；SHARD_GRAD_OP（对标ZeRO2）；FULL_SHARD （对标ZeRO3）。